# Building Energy Explorer: executed evidence walkthrough

Question: how well do mean prediction, Ridge and Random Forest predict heating/cooling loads for shapes absent from training?

One row is one simulated configuration, not a measured household. This notebook verifies and explains the saved experiment. Training and tuning live in `src/train.py`; run that script for a full replication. The following cells read real source data and saved results and can be rerun without hidden notebook state. No new models are selected here.

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
root = Path.cwd()
if not (root / "src/train.py").exists():
    root = root.parent
assert (root / "src/train.py").exists(), "Run from repository root or notebooks/"
sys.path.insert(0, str(root / "src"))
from verify_results import verify
raw = pd.read_excel(root / "data/ENB2012_data.xlsx")
print("Source rows/columns:", raw.shape)
print("Missing source values:", int(raw.iloc[:, :10].isna().sum().sum()))
print("Duplicate source rows:", int(raw.iloc[:, :10].duplicated().sum()))

Source rows/columns: (768, 10)
Missing source values: 0
Duplicate source rows: 0


## Grouping and leakage prevention
Geometry columns X1–X5 define 12 groups. Every group stays together for both development/test splitting and validation. A random row split would answer a different question, allowing familiar geometry on both sides. Scaling and categorical encoding are fitted inside training pipelines; the targets and IDs are excluded as predictors.

In [2]:
split = pd.read_csv(root / "results/split_assignments.csv")
folds = pd.read_csv(root / "results/cv_assignments.csv")
print(split.groupby("split").agg(rows=("row_id", "size"), shapes=("shape_id", "nunique")).to_string())
print("Validation shapes by fold:")
print(folds.groupby("validation_fold").shape_id.unique().to_string())
print("Selected settings:")
print((root / "results/best_parameters.json").read_text())

             rows  shapes
split                    
development   576       9
test          192       3
Validation shapes by fold:
validation_fold
0    [11, 6, 3]
1     [8, 5, 2]
2     [7, 4, 1]
Selected settings:
{
  "heating_load/baseline": {},
  "heating_load/ridge": {
    "model__alpha": 100
  },
  "heating_load/forest": {
    "model__max_depth": 5,
    "model__min_samples_leaf": 1
  },
  "cooling_load/baseline": {},
  "cooling_load/ridge": {
    "model__alpha": 100
  },
  "cooling_load/forest": {
    "model__max_depth": 5,
    "model__min_samples_leaf": 1
  }
}


## Full result comparison
MAE is mean absolute error, and RMSE gives greater weight to large errors. Both remain in source target units. Validation fold results were used for tuning, so they are descriptive rather than independent estimates. A mean baseline minimises squared training error; a median would minimise absolute training error.

In [3]:
metrics = pd.read_csv(root / "results/test_metrics.csv")
print(metrics.round(6).to_string(index=False))
for target in metrics.target.unique():
    m = metrics[metrics.target == target].set_index("model")
    gain = 100 * (1 - m.loc["ridge", "mae"] / m.loc["baseline", "mae"])
    print(f"{target}: Ridge reduces baseline MAE by {gain:.1f}%")

      target    model      mae     rmse
heating_load baseline 8.267684 8.877692
heating_load    ridge 2.463790 2.754134
heating_load   forest 3.329996 3.606803
cooling_load baseline 8.554783 9.093014
cooling_load    ridge 3.527426 4.133074
cooling_load   forest 4.343009 4.814943
heating_load: Ridge reduces baseline MAE by 70.2%
cooling_load: Ridge reduces baseline MAE by 58.8%


## Per-shape failures and coverage
Ridge is best overall but does not win on shape 0; the forest is better there for both targets. Geometry 0 has compactness and surface area outside development coverage. This describes a plausible extrapolation challenge, not a proven causal explanation. Neither grouping nor this inspection validates real-building performance.

In [4]:
print(pd.read_csv(root / "results/per_shape_metrics.csv").round(6).to_string(index=False))
development = raw.iloc[split.loc[split.split == "development", "row_id"]]
print("Development geometry ranges:")
print(development[["X1", "X2", "X3", "X4", "X5"]].agg(["min", "max"]).to_string())
large = pd.read_csv(root / "results/largest_errors.csv")
print("Largest errors for learned models:")
print(large[large.model != "baseline"].groupby(["target", "model"], sort=False).head(1).round(6).to_string(index=False))

      target    model  shape_id       mae
heating_load baseline         0  7.185774
heating_load baseline         9  7.357858
heating_load baseline        10 10.259420
heating_load    ridge         0  3.693841
heating_load    ridge         9  1.711140
heating_load    ridge        10  1.986388
heating_load   forest         0  2.335579
heating_load   forest         9  2.556210
heating_load   forest        10  5.098199
cooling_load baseline         0  8.653299
cooling_load baseline         9  7.087630
cooling_load baseline        10  9.923420
cooling_load    ridge         0  5.606437
cooling_load    ridge         9  2.566784
cooling_load    ridge        10  2.409056
cooling_load   forest         0  4.975359
cooling_load   forest         9  2.895140
cooling_load   forest        10  5.158527
Development geometry ranges:
       X1     X2     X3      X4   X5
min  0.64  514.5  245.0  110.25  3.5
max  0.98  784.0  416.5  220.50  7.0
Largest errors for learned models:
 row_id  shape_id  relative

## Independent artifact checks
The checks below recompute errors from row-level predictions, verify their target alignment with the Excel source, confirm the baseline is the development mean, and confirm shape and fold separation. They are consistency checks, not a second model replication or proof of student review.

## Fixed-model interpretation
The predefined Ridge models are refitted on development shapes with alpha 100. Their predictions match the saved test predictions. Coefficients describe conditional model associations, not causal or unique feature importance. No retuning or ablation is performed.

In [5]:
from interpret_models import interpret
interpret()
coefficients = pd.read_csv(root / "results/ridge_coefficients.csv")
print(coefficients.round(6).to_string(index=False))


Fixed Ridge refits match saved test predictions; coefficients exported.
      target                feature                   kind  coefficient
heating_load   relative_compactness standardised numerical    -0.175936
heating_load           surface_area standardised numerical    -0.817620
heating_load              wall_area standardised numerical     3.069374
heating_load              roof_area standardised numerical    -2.289119
heating_load         overall_height standardised numerical     4.215693
heating_load           glazing_area standardised numerical     2.141980
heating_load          orientation_2     category indicator     0.009943
heating_load          orientation_3     category indicator     0.026008
heating_load          orientation_4     category indicator    -0.031385
heating_load          orientation_5     category indicator    -0.004566
heating_load glazing_distribution_0     category indicator    -1.033592
heating_load glazing_distribution_1     category indicator     0

In [6]:
verification = verify()
assert verification["status"] == "passed"

{
  "status": "passed",
  "checks": [
    "source SHA-256 matches audit",
    "768 complete rows and original geometry grouping verified",
    "576 development / 192 test rows; no shape overlap",
    "three validation folds with shape separation",
    "prediction targets match source rows",
    "baseline equals development target mean",
    "all six MAE and RMSE values recomputed",
    "per-shape MAE recomputed",
    "website JSON matches saved predictions"
  ],
  "scope": "Artifact consistency checks, not a claim of student understanding or independent model replication."
}


## Conclusion and scope
Both learned models beat the predefined baseline; Ridge has lower aggregate MAE for both targets on the saved three-shape test. Shape-specific exceptions and fold variation limit any broader claim. The data is simulated and contains only 12 geometries. No causal effects, real-building validity, significance result or calibrated uncertainty are established.

## Reproduce from scratch
From the project root, install `requirements.txt`, run `python src/train.py`, then `python src/verify_results.py`. Training regenerates the report and exports. Serve `docs/` using `python -m http.server 8000 --directory docs`.

## Sources and AI disclosure
Dataset: Tsanas & Xifara (2012), UCI Energy Efficiency, DOI https://doi.org/10.24432/C51307, CC BY 4.0. Methods use scikit-learn; see CITATIONS.md for software sources. ChatGPT/Codex assisted with planning, implementation and explanation. The student must independently review and understand all submitted work; see AI_USE.md.